# Tutorials

## 🚀 Quick start - install

To install `covmats`, the easiest way is through `pip`:

```bash
    pip install covmats
```
Or alternatively using `conda`

```bash
    conda install covmats
```

You might also clone the repository and install from source

```bash
    pip install -e .
```

Once the installation is done, we can start using covariance matrices. There are various representations:
- CovarianceMatrix
- CovViaDiag
- CovViaDense
- CovViaCholesky
- CovViaEigendecomposition
- CovViaEnsemble
- CovViaFFT
- CovViaHierarchical
- CovViaPrecision
- CovViaPSD
- CovViaSparseCholesky
- CovViaSparsePrecision


Convention, Q for the precision and Sigma for 

- Let's start with the dense version

In [1]:
import scipy as sp
import covmats

## Diagonal matrix

Let's start with the simple case of a diagonal matrix. 

In [2]:
import numpy as np

d = [1, 2, 3]
A33 = np.diag(d)  # a diagonal covariance matrix
x = [4, -2, 5]  # a point of interest
dist = sp.stats.multivariate_normal(mean=[0, 0, 0], cov=A33)
dist.pdf(x)

np.float64(4.9595685102808205e-08)

It is compatible with the stats API from scipy since the base class inherit from `Covariance`.

In [3]:
cov_diag33 = covmats.CovarianceMatrix.from_diagonal(d)
dist = sp.stats.multivariate_normal(mean=[0, 0, 0], cov=cov_diag33)
dist.pdf(x)

array([4.38559708e-12, 3.37164352e-07, 1.43365931e-05])

In [4]:
cov_diag33.rank, cov_diag33.log_pdet, cov_diag33.get_trace()

(np.int64(3), np.float64(1.791759469228055), 6.0)

- It also behaves as a Linearoperator, supporting matrix-vector, matrix-matrix and solve operations

In [5]:
v = np.array([1.0, 2.0, 3.0])
cov_diag33 @ v

array([1., 4., 9.])

In [7]:
np.linalg.inv(A33) @ v

array([1., 1., 1.])

In [ ]:
cov_diag33.solve(v)

array([1., 1., 1.])

In [ ]:
# product with a matrix (3, 2)
V = np.array([[1.0, 2.0, 3.0], [1.0, 2.0, 3.0]]).T
cov_diag33 @ V

array([[1., 1.],
       [4., 4.],
       [9., 9.]])

In [ ]:
cov_diag33.solve(V)

array([[1., 1.],
       [1., 1.],
       [1., 1.]])

In [ ]:
cov_diag33.whiten(v)

array([1.        , 1.41421356, 1.73205081])

In [ ]:
V

array([[1., 1.],
       [2., 2.],
       [3., 3.]])

In [ ]:
cov_diag33.whiten(V).shape

(3, 2)

## Dense covariance matrix

With a dense covariance matrix, it is possible to compute the rank and the determinant directly.

In [8]:
cov_dense33 = covmats.CovarianceMatrix.from_dense(A33)
cov_dense33.rank, cov_dense33.log_pdet, cov_dense33.get_trace()

(np.int64(3), np.float64(1.791759469228055), 6)

It is also possible to perform classic operations such as matrix-vector multiplications, matrix-matrix or even solving systems

In [10]:
v = np.array([1.0, 2.0, 3.0])
cov_dense33 @ v, cov_dense33.solve(v)

(array([1., 4., 9.]), array([1., 1., 1.]))

In [11]:
# product with a matrix (3, 2)
V = np.array([[1.0, 2.0, 3.0], [1.0, 2.0, 3.0]]).T
cov_dense33 @ V, cov_dense33.solve(V)

(array([[1., 1.],
        [4., 4.],
        [9., 9.]]),
 array([[1., 1.],
        [1., 1.],
        [1., 1.]]))

How ever, other operations such as whitthening, etc. requires to decompose the matrix

Or for an another example.

In [26]:
rng = np.random.default_rng(2026)
n = 5
A55 = rng.random(size=(n, n))
A55 = A55 @ A55.T  # make the covariance symmetric positive definite
x = rng.random(size=n)

cov_dense55 = covmats.CovarianceMatrix.from_dense(A55)
cov_dense55.rank, cov_dense55.log_pdet, cov_dense55.get_trace()

(np.int64(5), np.float64(-10.385086971402448), 8.160290791028103)

In [27]:
v = np.array([4.5, -2.0, 3.1, 0.0, 2.7])
cov_dense55 @ v

array([ 8.84197694, 11.37527262, 15.71712939,  8.77406372,  9.49875718])

In [28]:
cov_dense55.solve(v)

array([ 306.83325703, -174.83589997, -126.25871758,  375.70748055,
       -210.03940781])

In [29]:
V = np.tile(v.reshape(-1, 1), 2)
cov_dense55 @ V

array([[ 8.84197694,  8.84197694],
       [11.37527262, 11.37527262],
       [15.71712939, 15.71712939],
       [ 8.77406372,  8.77406372],
       [ 9.49875718,  9.49875718]])

In [30]:
np.linalg.inv(cov_dense55.covariance) @ V

array([[ 306.83325703,  306.83325703],
       [-174.83589997, -174.83589997],
       [-126.25871758, -126.25871758],
       [ 375.70748055,  375.70748055],
       [-210.03940781, -210.03940781]])

- The output should be identical

In [31]:
cov_dense55.solve(V)

array([[ 306.83325703,  306.83325703],
       [-174.83589997, -174.83589997],
       [-126.25871758, -126.25871758],
       [ 375.70748055,  375.70748055],
       [-210.03940781, -210.03940781]])

In [32]:
# Perform the Cholesky decomposition of ``A`` and create the `Covariance` object.
L = np.linalg.cholesky(A)
cov_cho55 = covmats.CovarianceMatrix.from_cholesky(L)
cov_cho55.rank, cov_dense55.log_pdet, cov_cho55.get_trace()

(np.int64(5), np.float64(-10.385086971402448), 8.160290791028103)

In [33]:
cov_cho55.shape == (5, 5)

True

In [34]:
v = np.array([4.5, -2.0, 3.1, 0.0, 2.7])
cov_cho55 @ v

array([ 8.84197694, 11.37527262, 15.71712939,  8.77406372,  9.49875718])

In [35]:
cov_cho55.solve(v)

array([ 306.83325703, -174.83589997, -126.25871758,  375.70748055,
       -210.03940781])

In [36]:
cov_cho55 @ V

array([[ 8.84197694,  8.84197694],
       [11.37527262, 11.37527262],
       [15.71712939, 15.71712939],
       [ 8.77406372,  8.77406372],
       [ 9.49875718,  9.49875718]])

In [37]:
cov_cho55.solve(V)

array([[ 306.83325703,  306.83325703],
       [-174.83589997, -174.83589997],
       [-126.25871758, -126.25871758],
       [ 375.70748055,  375.70748055],
       [-210.03940781, -210.03940781]])

In [38]:
cov_cho55.get_diagonal()

array([0.92308324, 1.99077116, 3.01746083, 1.1263966 , 1.10257896])

## Working with precision

It is also possible to work with the inverse of the covariance matrix, namely the precision matrix

## Eigen 

eigen_factorize_cov_mat,
generate_dense_matrix,
get_explained_var,
get_matrix_eigen_factorization,

## SVD

## Kernel based Covariances

Lib such as gstools, gstlearn, etc. provide kernels

### FFT

### Hirearchical

## Ensemble of realizations

## Sparse precision and cholesky

- Talk about SPDE, large scale applications